<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde
!pip install optuna

In [2]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import optuna
import plotly.express as px
import itertools

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
dde.config.set_default_float("float64")

Set the default float type to float64


In [4]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [5]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [6]:
def exact(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

In [7]:
iterations = 10000
epsilon = 1e-4

In [8]:
def output(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

In [9]:
XT_test = geomtime.uniform_points(5000)
y_true = exact(XT_test)

In [10]:
fig = px.scatter(
    x=XT_test[:, 0],
    y=XT_test[:, 1],
    labels={"x": "Координата x", "y": "Время t"}
)

fig.add_shape(
    type="rect",
    x0=XT_test[:, 0].min(), y0=XT_test[:, 1].min(),
    x1=XT_test[:, 0].max(), y1=XT_test[:, 1].max(),
    line=dict(color="blue", width=2),
    fillcolor="rgba(0, 0, 255, 0.05)",
)

fig.update_layout(template="plotly_white")
fig.update_traces(marker=dict(size=6, color="red"))
fig.show()

In [11]:
search_space = {"layers": [2, 4, 6], "neurons": [20, 32, 64], "num_domain": [500, 1000, 2000]}
keys = search_space.keys()
combinations = itertools.product(*search_space.values())

In [12]:
dde.optimizers.config.set_LBFGS_options(maxiter=5000)
for values in combinations:
  params = dict(zip(keys, values))

  dde.config.set_random_seed(42)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=params["num_domain"], train_distribution="uniform")

  net = dde.nn.FNN([2] + [params["neurons"]] * params["layers"] + [1], "tanh", "Glorot normal")
  net.apply_output_transform(output)

  model = dde.Model(data, net)
  model.compile("adam", lr=1e-3)
  model.train(iterations=iterations)

  model.compile("L-BFGS")
  model.train()

  y_pred = model.predict(XT_test)
  l2_error = dde.metrics.l2_relative_error(y_true, y_pred)

  print(f"{params} | L2 Error: {l2_error:.4e}")

  if l2_error < epsilon:
    break

Compiling model...
'compile' took 4.367733 s

Training model...

Step      Train loss    Test loss     Test metric
0         [4.24e+00]    [4.24e+00]    []  
1000      [7.71e-04]    [7.71e-04]    []  
2000      [2.44e-04]    [2.44e-04]    []  
3000      [8.63e-05]    [8.63e-05]    []  
4000      [3.26e-05]    [3.26e-05]    []  
5000      [1.88e-05]    [1.88e-05]    []  
6000      [1.39e-05]    [1.39e-05]    []  
7000      [1.11e-05]    [1.11e-05]    []  
8000      [9.49e-06]    [9.49e-06]    []  
9000      [8.23e-06]    [8.23e-06]    []  
10000     [7.18e-06]    [7.18e-06]    []  

Best model at step 10000:
  train loss: 7.18e-06
  test loss: 7.18e-06
  test metric: []

'train' took 74.384145 s

Compiling model...
'compile' took 0.000440 s

Training model...

Step      Train loss    Test loss     Test metric
10000     [7.18e-06]    [7.18e-06]    []  
10889     [5.93e-07]    [5.93e-07]    []  
11722     [5.87e-07]    [5.87e-07]    []  
12555     [5.81e-07]    [5.81e-07]    []  
13388   